## Claude Edit Log

Tracks whether Claude Code's edits have actually landed in this Kaggle-hosted copy (not just the local repo file) -- updated every time a new fix is pushed here.

- **v1** (2026-08-25 01:39 UTC): Fixed regressions/new bugs specific to this copy vs. the locally-fixed notebook -- `torchaudio` uninstall was commented out and `torchvision` uninstall was missing entirely (cell 6); the speaker-split filter was missing `input_columns=["speaker_id"]` (cell 15, regressed back to the "stuck at Filter: 0%" bug); the row-validity filter still had `num_proc=1` (cell 17, the forked-Pool deadlock bug); a cell doing a direct `import torchaudio` before the availability-monkeypatch ran was deleted (crashed regardless of the patch, and was redundant with the next cell's imports).

- **v2** (2026-08-25): Appended a standalone evaluation section at the very bottom (after "Closing Remarks") -- loads the fine-tuned model fresh from the Hugging Face Hub (`troxyz1268/whisper-small-bisaya`), NOT the in-kernel `model`/`processor`/`trainer` objects from the training cells above, so it gives the same result whether or not training actually ran in this session. Restricted to the same 19-utterance held-out test set `evaluate_kaldi.ipynb`/`evaluate_elevenlabs.ipynb` use locally (recomputes the speaker-independent split, seed 42, directly from the corpus -- doesn't depend on the training cells' own `test_speakers` variable). Exports `output/whisper/results.csv` under `/kaggle/working/` in the exact schema the local `evaluate_whisper.ipynb` produces, for direct download into the local `output/whisper/` folder -- `compare.ipynb` needs no changes to read it.

- **v3** (2026-08-25): Fixed a CUDA crash in the eval section's model-loading cell (`4e5d33bd`) -- `torch.cuda.is_available()` only confirms the driver sees a GPU, not that the installed torch build has kernels compiled for it; running just the eval cells (skipping the earlier torch/torchaudio/torchvision reinstall + kernel-restart cells, since training is assumed done) left Kaggle's default preinstalled torch active, which threw `CUDA error: no kernel image is available for execution on the device` on the T4 the moment `pipeline()` tried to actually use it. Device selection now runs a real tiny op on `cuda` and falls back to CPU automatically if that fails, instead of trusting `is_available()` alone.

# Fine-Tune Whisper on the Bisaya Speech Corpus (Kaggle Notebook)

This is a Kaggle-notebook adaptation of Hugging Face's
[Fine-Tune Whisper for Multilingual ASR](https://huggingface.co/blog/fine-tune-whisper)
tutorial, pointed at this project's own Bisaya (Cebuano) speech corpus
instead of the tutorial's original Hindi/Common Voice example. It trains
a third ASR system for this project's benchmark, alongside the Kaldi
HMM-GMM model (`train_kaldi.ipynb`) and ElevenLabs Scribe
(`evaluate_elevenlabs.ipynb`) -- see the repo's `README.md` for the rest
of that pipeline. This notebook is standalone: it doesn't feed into
`compare.ipynb` automatically.

**Differences from the original Colab tutorial**, besides the dataset:
- GPU/internet setup, Hugging Face auth, and long-run persistence all use
  Kaggle's own mechanisms instead of Colab's (Section "Kaggle Setup" below).
- The corpus has to be **uploaded to Kaggle as a Dataset first** (Section
  "Upload the Corpus to Kaggle" below) -- Kaggle notebooks can't read your
  local filesystem the way Colab can read from Google Drive or a direct
  download.
- The train/test split reuses the exact same speaker-independent split
  (by speaker, ~80/20, fixed seed 42) that `train_kaldi.ipynb` uses, so
  Whisper's held-out test speakers match Kaldi's -- a prerequisite for any
  later three-way comparison.
- Whisper has no dedicated Cebuano/Bisaya language token (~99 languages
  are supported; Bisaya isn't one). This notebook uses `"tl"` (Tagalog --
  Whisper's closest Philippine-language code) for the tokenizer/
  generation language conditioning -- a real approximation, not an exact
  match, called out again where it's used.

**Fixed after a collapsed-generation run** (repeated `<|fr|>` tokens, 100%
WER): the decoder prompt is now forced deterministically via
`forced_decoder_ids` instead of relying on generate()-time language
auto-detection, and the learning rate is back to a full-fine-tuning-scale
`1e-5` -- the notebook installs `peft` and named its output dir
`-lora`, but never actually wrapped the model in a LoRA adapter, so it was
silently doing full-parameter fine-tuning at a LoRA-scale (100x too high)
learning rate. See the note on the training-configuration cell below if
you want to add real LoRA instead of reverting the learning rate.

## Kaggle Setup

1. **Enable a GPU**: notebook Settings (right sidebar) -> Accelerator ->
   GPU T4 x2 (or P100/other, if available). Kaggle currently grants a
   weekly GPU-hour quota per account -- check your remaining quota in
   Settings before starting a long run.
2. **Enable internet access**: Settings -> Internet -> On. Required for
   `pip install`, downloading the pretrained Whisper checkpoint, and
   pushing to the Hugging Face Hub.
3. **Attach the corpus dataset**: see "Upload the Corpus to Kaggle" below
   -- do this once, then attach it via Add Input on every notebook that
   needs it.
4. **Add your Hugging Face token as a Kaggle Secret**: Add-ons -> Secrets
   -> add a secret named `HF_TOKEN` with a Hugging Face
   [write access token](https://huggingface.co/settings/tokens) as the
   value. Used in "Hugging Face Authentication" below instead of the
   interactive `notebook_login()` widget, so the notebook can run
   unattended (Save & Run All).

## Upload the Corpus to Kaggle

One-time step, done outside this notebook, before it can run:

1. Go to [kaggle.com/datasets](https://www.kaggle.com/datasets) -> **New
   Dataset**.
2. Upload every file under this project's `data/bisaya_audio/` (the
   Parquet shards) -- drag-and-drop in the browser, or use the
   [Kaggle API](https://www.kaggle.com/docs/api) from the machine that
   has the corpus locally:
   ```bash
   pip install kaggle
   # ~/.kaggle/kaggle.json holds your API credentials (Kaggle account ->
   # Settings -> Create New Token)
   kaggle datasets init -p data/bisaya_audio
   # edit the generated dataset-metadata.json: set a title/id, e.g.
   #   "id": "your-kaggle-username/bisaya-audio-corpus"
   kaggle datasets create -p data/bisaya_audio
   ```
3. In this notebook (or any Kaggle notebook that needs the corpus): **Add
   Input** (right sidebar) -> search for the dataset you just created ->
   Add. It appears under `/kaggle/input/<dataset-slug>/`.
4. Set `CORPUS_DIR` in the next cell to match wherever your Parquet files
   actually land under `/kaggle/input/`.

This corpus is not public -- keep the uploaded Kaggle Dataset **Private**
unless you have the right to publish it.

## Prepare Environment

In [ ]:
!nvidia-smi


In [ ]:
# datasets pinned to <4.0 (tested: 3.6.0) -- 4.0+ made torchcodec a hard
# requirement for decoding Audio columns, and torchcodec needs an FFmpeg
# build whose shared libs (libavutil.so.57-60) aren't present on this
# Kaggle image, causing "Could not load libtorchcodec" the moment any
# audio is touched. 3.6.0 uses the classic soundfile-based decoder
# (returns {"array", "sampling_rate", "path"}), no FFmpeg/torchcodec
# involved at all.
pip uninstall -y -q torchaudio

# Same root cause, different package: torchvision is also mismatched
# against this Kaggle image's torch build. transformers opportunistically
# imports torchvision while building WhisperProcessor, and torchvision's
# custom-op registration crashes on the mismatch ("operator
# torchvision::nms does not exist"), taking down the whole import chain.
# Not needed here -- this notebook never touches images/video.
pip uninstall -y -q torchvision

!pip install -q --force-reinstall "numpy<2.0.0" "scipy<1.14.0" "protobuf>=5.29.1,<6.0.0" "datasets<4.0" pyarrow transformers accelerate peft librosa soundfile

In [ ]:
import os

print("Environment setup complete! Restarting kernel...")
os._exit(0)


In [ ]:
import os, subprocess

print("Installing system FFmpeg and setting audio decoding backend...")

# 1. Install system FFmpeg libraries required by C-extension audio decoders
subprocess.run(
    "apt-get update -qq && apt-get install -y -qq ffmpeg",
    shell=True,
    check=True,
)

# 2. Disable torchcodec explicitly so datasets uses soundfile/librosa
os.environ["HF_DATASETS_DISABLE_TORCHCODEC"] = "1"

print("FFmpeg installed and torchcodec disabled!")


### Hugging Face Authentication

Uses the `HF_TOKEN` Kaggle Secret set up above, rather than the
interactive `notebook_login()` widget the original Colab tutorial uses --
this keeps the notebook runnable unattended via Kaggle's **Save & Run
All (Commit)**.

In [ ]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

hf_token = UserSecretsClient().get_secret("HF_TOKEN")

# Login and expose as environment variable for all Hugging Face libraries
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token


## Load Dataset

Reads every Parquet shard in `CORPUS_DIR` via 🤗 `datasets` directly --
the corpus's `audio` column is already stored in `datasets`' own Audio
struct format (`{bytes, path}`, with the feature type recorded in the
Parquet file's own schema metadata), so `load_dataset("parquet", ...)`
decodes it automatically, the same way it would for a published HF
dataset like the tutorial's Common Voice.

In [ ]:
from pathlib import Path

# Adjust to match the slug/path your attached Kaggle Dataset actually uses.
CORPUS_DIR = Path("/kaggle/input/datasets/troymerales/bisaya-audio")

parquet_files = sorted(str(p) for p in CORPUS_DIR.glob("*.parquet"))
assert parquet_files, f"No Parquet files found under {CORPUS_DIR} -- check the dataset is attached (Add Input)."
print(f"Found {len(parquet_files)} Parquet shard(s)")


In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("parquet", data_files=parquet_files, split="train")
print(raw_dataset)


### Speaker-Independent Train/Test Split

Same split `train_kaldi.ipynb` Section 8 uses -- by speaker (not
utterance), ~80/20, fixed seed 42 -- so this notebook's held-out test
speakers are the same ones Kaldi never trained on. Reused verbatim rather
than re-derived, so the split stays identical if this cell or
`train_kaldi.ipynb`'s changes independently.

`test_fraction` is back to `0.2` here (was `0.1`) -- on a ~77-utterance
corpus a 10% test split can leave only one or two speakers held out,
which makes eval WER extremely high-variance (one bad utterance can swing
it by tens of points) and breaks parity with `train_kaldi.ipynb`'s own
80/20 split.

In [ ]:
from datasets import DatasetDict
import pandas as pd

SPLIT_SEED = 42


def speaker_independent_split(speakers, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(speakers)
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])
    assert train_speakers.isdisjoint(test_speakers)
    return train_speakers, test_speakers


all_speakers = set(raw_dataset.unique("speaker_id"))
train_speakers, test_speakers = speaker_independent_split(all_speakers)

# input_columns=["speaker_id"] -- without it, .filter() touches every
# column (including "audio") just to size the batch internally, which
# forces a full decode of all ~90 multi-minute audio files even though
# the lambda only needs speaker_id -- confirmed locally: ~21s vs ~0s for
# the same filter. This is what "stuck at Filter: 0%" actually was.
bisaya = DatasetDict(
    {
        "train": raw_dataset.filter(
            lambda spk: spk in train_speakers, input_columns=["speaker_id"]
        ),
        "test": raw_dataset.filter(
            lambda spk: spk in test_speakers, input_columns=["speaker_id"]
        ),
    }
)

print(f"seed = {SPLIT_SEED}")
print(f"train: {len(train_speakers)} speakers, {len(bisaya['train'])} audio files")
print(f"test:  {len(test_speakers)} speakers, {len(bisaya['test'])} audio files")

# On a very small corpus a couple of unlucky speakers can leave a split
# empty -- fail loudly here instead of getting a meaningless WER (or a
# crash inside evaluate's wer metric) much later in the notebook.
assert len(bisaya["train"]) > 0, "train split is empty -- check speaker_id values in the corpus"
assert len(bisaya["test"]) > 0, "test split is empty -- too few speakers for this test_fraction"


In [ ]:
# Keep audio, transcript, and words (word-level timestamps, needed by
# prepare_dataset below to chunk long utterances to Whisper's 30s window)
keep_cols = {"audio", "transcript", "words"}
bisaya = bisaya.remove_columns(
    [c for c in bisaya["train"].column_names if c not in keep_cols]
)
print("Columns after pruning:", bisaya["train"].column_names)
# Output should be: ['audio', 'transcript', 'words']


In [ ]:
def _get_audio_array_sr(audio):
    """Returns (array, sampling_rate) regardless of which form `datasets`
    decoded this into: the classic dict ({"array": ..., "sampling_rate": ...})
    or a torchcodec AudioDecoder object (current datasets versions -- confirmed
    HF_DATASETS_DISABLE_TORCHCODEC does NOT prevent this)."""
    if isinstance(audio, dict):
        return audio.get("array"), audio.get("sampling_rate")
    samples = audio.get_all_samples()  # torchcodec AudioDecoder path
    array = samples.data.mean(dim=0).numpy()  # [channels, samples] -> mono
    return array, samples.sample_rate


# Drop rows with a missing/empty transcript or missing/empty audio.
# Checks both audio representations via _get_audio_array_sr() above --
# this corpus's datasets/torchcodec version decodes "audio" into an
# AudioDecoder object (not a plain dict), so a bytes/dict-only check drops
# every row regardless of content (confirmed against the real corpus).
def _is_valid_row(example):
    transcript = (example.get("transcript") or "").strip()
    if not transcript:
        return False
    try:
        array, sr = _get_audio_array_sr(example.get("audio"))
    except Exception:
        return False
    return array is not None and bool(sr) and len(array) > 0


before_counts = {split: len(bisaya[split]) for split in bisaya}
# No num_proc -- in this datasets version, even num_proc=1 routes through
# a multiprocess Pool (forked), which deadlocks with torchcodec's native
# ffmpeg-backed audio decoder. Omitting it runs single-process, no fork.
# load_from_cache_file=False -- avoids datasets silently reusing a stale
# on-disk cache from an earlier version of _is_valid_row.
bisaya = bisaya.filter(_is_valid_row, load_from_cache_file=False)
after_counts = {split: len(bisaya[split]) for split in bisaya}
print(f"null/corrupted rows dropped: {before_counts} -> {after_counts}")

assert len(bisaya["train"]) > 0, "no valid training rows left after filtering"
assert len(bisaya["test"]) > 0, "no valid test rows left after filtering -- eval/WER will be meaningless"


## Prepare Feature Extractor, Tokenizer and Data

### Load WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

`language="tl"` (Tagalog) is Whisper's closest built-in Philippine-language
code -- there is no Cebuano/Bisaya entry in Whisper's ~99-language list.
This is an approximation the model was not specifically designed for;
treat any language-conditioning benefit from it as best-effort, not exact.
The short ISO code is used instead of the full name `"Tagalog"` so it's
unambiguous everywhere it's reused below (tokenizer, processor, and the
forced decoder prompt).

In [ ]:
import transformers.utils.import_utils as _iu
_iu.is_torchaudio_available = lambda: False
_iu.is_torchvision_available = lambda: False

from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

MODEL_CHECKPOINT = "openai/whisper-small"
LANGUAGE = "tl"  # ISO code for Tagalog -- Whisper's closest built-in language to Bisaya/Cebuano (see note above)

feature_extractor = WhisperFeatureExtractor.from_pretrained(MODEL_CHECKPOINT)
tokenizer = WhisperTokenizer.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")
processor = WhisperProcessor.from_pretrained(MODEL_CHECKPOINT, language=LANGUAGE, task="transcribe")

expected_prefix = ["<|startoftranscript|>", f"<|{LANGUAGE}|>", "<|transcribe|>", "<|notimestamps|>"]
actual_prefix = tokenizer.convert_ids_to_tokens(tokenizer.prefix_tokens)
assert actual_prefix == expected_prefix, (
    f"tokenizer prefix tokens are {actual_prefix}, expected {expected_prefix}"
)
print("decoder prefix tokens:", actual_prefix)


### Prepare Data

In [ ]:
print(bisaya["train"][0]["transcript"])


In [ ]:
print(bisaya.column_names)


In [ ]:
from datasets import Audio

# 1. Ensure 16kHz audio sampling
bisaya = bisaya.cast_column("audio", Audio(sampling_rate=16000))

# Every utterance in this corpus is longer than Whisper's fixed 30-second
# input window (median ~120s, max ~300s -- confirmed against the full
# corpus). Truncating audio to 30s while keeping the FULL transcript as the
# label (the previous version of this cell) teaches the model to produce
# text far beyond what it can actually hear -- that's why predictions
# tracked the reference for the first ~20-30s of content, then degraded or
# stopped. Fix: use this corpus's word-level timestamps (`words`: list of
# {text, start, end}) to split each utterance into <=28s chunks with
# correctly matched audio/text, and train on those instead.
MAX_CHUNK_SECONDS = 28  # safety margin under Whisper's 30s window


def _chunk_words(words, max_seconds=MAX_CHUNK_SECONDS):
    chunks, cur, cur_start = [], [], None
    for w in words:
        if cur_start is None:
            cur_start = w["start"]
        if w["end"] - cur_start > max_seconds and cur:
            chunks.append(cur)
            cur, cur_start = [], w["start"]
        cur.append(w)
    if cur:
        chunks.append(cur)
    return chunks


def prepare_dataset(examples):
    # Batched: emits a variable number of chunk examples per input
    # utterance (not exactly one), so this must run with batched=True.
    out_features, out_labels = [], []

    for i in range(len(examples["audio"])):
        array, sr = _get_audio_array_sr(examples["audio"][i])
        words = examples["words"][i]

        if not words:
            # No word-level alignment available -- fall back to a single
            # chunk from the start of the audio (will misalign for
            # utterances over MAX_CHUNK_SECONDS, but there's no timestamp
            # data to chunk against for this row).
            segments = [(array[: int(MAX_CHUNK_SECONDS * sr)], examples["transcript"][i])]
        else:
            segments = []
            for chunk in _chunk_words(words):
                start, end = chunk[0]["start"], chunk[-1]["end"]
                seg_array = array[int(start * sr): int(end * sr)]
                seg_text = " ".join(w["text"] for w in chunk)
                segments.append((seg_array, seg_text))

        for seg_array, seg_text in segments:
            if len(seg_array) == 0 or not seg_text.strip():
                continue
            out_features.append(
                feature_extractor(seg_array, sampling_rate=sr).input_features[0]
            )
            out_labels.append(
                tokenizer(seg_text, max_length=448, truncation=True).input_ids
            )

    return {"input_features": out_features, "labels": out_labels}


# No num_proc -- in this datasets version, even num_proc=1 routes through
# a multiprocess Pool (forked), which deadlocks with torchcodec's native
# ffmpeg-backed audio decoder (this is what "stuck at 0%" was). Omitting
# it runs single-process, no fork.
bisaya = bisaya.map(
    prepare_dataset,
    batched=True,
    batch_size=4,
    remove_columns=bisaya["train"].column_names,
    load_from_cache_file=False,
)

print("Processed dataset columns:", bisaya["train"].column_names)
print("Chunked example counts:", {split: len(bisaya[split]) for split in bisaya})
# Expect noticeably MORE rows than before (each multi-minute utterance
# becomes several <=28s chunks) -- more training signal, not less.

In [ ]:
# Size generation_max_length off the actual tokenized labels instead of
# the fixed 225 guess -- used by training_args below, capped at Whisper's
# decoder position limit (448) that prepare_dataset() already truncates to.
max_label_len = max(
    max(len(x) for x in bisaya["train"]["labels"]),
    max(len(x) for x in bisaya["test"]["labels"]),
)
GENERATION_MAX_LENGTH = min(448, max_label_len + 10)
print(f"longest tokenized label = {max_label_len} tokens -> generation_max_length = {GENERATION_MAX_LENGTH}")


## Training and Evaluation

Same 🤗 Trainer-based pipeline as the original tutorial: load a
pretrained checkpoint, define a data collator, define the WER metric,
configure and run training.

### Load a Pre-Trained Checkpoint

In [ ]:
from transformers import WhisperForConditionalGeneration
import torch
import gc

model = WhisperForConditionalGeneration.from_pretrained(MODEL_CHECKPOINT)

# Deterministically force the first 4 decoder tokens
# (<|startoftranscript|>, <|tl|>, <|transcribe|>, <|notimestamps|>) instead
# of leaving generate() to auto-detect language from the encoder output.
# On a small fine-tuned model that auto-detection is unreliable -- it's the
# direct cause of the reported failure mode: generation collapsing into a
# repeated wrong-language special token (e.g. <|fr|>), which decodes to an
# empty string and scores 100% WER. This is saved into the model's
# generation_config.json by save_pretrained()/push_to_hub(), so it also
# applies automatically to any later pipeline()/deploy inference -- no need
# to pass forced_decoder_ids by hand at generate()-time anymore.
model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
    language=LANGUAGE, task="transcribe"
)

torch.cuda.empty_cache()
gc.collect()


### Define a Data Collator

In [ ]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Pad pre-extracted log-Mel input_features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # Pad label token sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding tokens in labels with -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Strip initial decoder start token if present
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch


data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)


### Evaluation Metrics

In [ ]:
pip install evaluate jiwer

In [ ]:
import evaluate
import jiwer
import numpy as np

# Load Word Error Rate metric
metric = evaluate.load("wer")


def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # Replace -100 in labels with pad_token_id so decoding doesn't crash
    label_ids = np.where(label_ids != -100, label_ids, tokenizer.pad_token_id)

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    # Guard against a still-collapsed model whose predictions decode to ""
    # (all special tokens) -- evaluate's wer metric divides by reference
    # length and raises on an empty reference, which would otherwise crash
    # trainer.evaluate() mid-training instead of just reporting a bad WER.
    pairs = [(p, l) for p, l in zip(pred_str, label_str) if l.strip()]
    if not pairs:
        return {"wer": 100.0}
    preds, labels = zip(*pairs)
    wer = 100 * metric.compute(predictions=list(preds), references=list(labels))
    return {"wer": wer}


### Define the Training Configuration

**Learning rate fixed: `1e-3` -> `1e-5`.** The pip-install cell installs
`peft` and `output_dir` was named `...-lora`, but no `LoraConfig`/
`get_peft_model()` call ever wraps the model anywhere in this notebook --
so training was doing full-parameter fine-tuning at a LoRA-scale learning
rate (100x too high for full fine-tuning). On a small pretrained
transformer that's a direct route to the reported collapse: weights blow
up within the first few steps, and generation degenerates into repeating
a single token. `1e-5` matches the original tutorial's full-fine-tuning
rate. (If you actually want LoRA -- a reasonable choice for a ~77-example
corpus, since it trains far fewer parameters and overfits less -- that's
a separate, deliberate change: wrap `model` with
`peft.get_peft_model(model, LoraConfig(...))` and call
`model.enable_input_require_grads()` before building the Trainer, then
the `1e-3` rate becomes appropriate again. Left out here since it wasn't
part of the current code and changes the training approach, not just
fixes a bug in it.)

`num_train_epochs=5` / `eval_strategy="epoch"` / `save_strategy="epoch"`
were already correctly sized for this small corpus -- no change needed
there.

`push_to_hub=True` is still the main persistence strategy on Kaggle, same
as the original tutorial recommends for Colab: Kaggle notebook sessions
are also ephemeral (working-directory contents don't survive past the
session unless explicitly saved), so periodic checkpoints pushed to the
Hub protect the run against an interrupted or killed session.

In [ ]:
print("Train dataset column names:", bisaya["train"].column_names)
print("First item keys:", bisaya["train"][0].keys())


In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-small-bisaya",  # was "...-lora" -- no LoRA adapter is actually applied, see note above
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,  # Effective batch size = 16
    learning_rate=1e-5,  # was 1e-3 (LoRA-scale) -- see note above, this is the fix for the reported collapse
    warmup_ratio=0.1,  # Uses 10% of total steps for warmup instead of a fixed 500 steps
    num_train_epochs=5,  # Loops through the ~3.14h train set 5 times (~75 total steps)
    gradient_checkpointing=True,
    fp16=True,
    eval_strategy="epoch",  # Evaluates WER once at the end of every epoch
    save_strategy="epoch",  # Saves a checkpoint once at the end of every epoch
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=GENERATION_MAX_LENGTH,  # computed from the actual tokenized labels, not guessed
    generation_num_beams=1,
    logging_steps=5,  # Reduced logging interval since total steps will be < 100
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True,
)


In [ ]:
from transformers import Seq2SeqTrainer, EarlyStoppingCallback

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=bisaya["train"],
    eval_dataset=bisaya["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
    # Stops the run once eval WER hasn't improved for 3 epochs, instead of
    # burning the rest of num_train_epochs after the model has collapsed.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)


### Training

**Running unattended on Kaggle:** unlike Colab (which needs the browser
tab open and a JS keep-alive trick to survive idle disconnects), Kaggle
runs a notebook as a real background batch job when you click **Save
Version -> Save & Run All (Commit)** -- you can close the browser and it
keeps training. Use interactive "Edit" mode only for iterating on the
early cells; switch to a commit run for the actual training job.

Watch your GPU-hour quota -- a run that hits Kaggle's session time limit
partway through is still recoverable via `push_to_hub`'s periodic
checkpoints (re-load with `from_pretrained` on your Hub repo and resume),
but budgeting the run to fit inside one session avoids that entirely.

In [ ]:
# Launch fine-tuning run
trainer.train()


In [ ]:
import torch
import gc
import pandas as pd

# Confirm the best checkpoint (already loaded via load_best_model_at_end)
# is actually usable BEFORE pushing anything to the Hub.
eval_metrics = trainer.evaluate()
print("Evaluation Results:", eval_metrics)
if eval_metrics.get("eval_wer", 100) >= 100:
    print(
        "WARNING: eval WER is 100% -- the model likely collapsed. "
        "Review the learning-rate/forced_decoder_ids fixes above before "
        "pushing to the Hub."
    )

# Side-by-side sample check -- forced_decoder_ids is now set globally on
# model.generation_config (see the model-loading cell), so plain
# model.generate() below already uses it; no per-call override needed.
sample_batch = bisaya["test"].select(range(min(5, len(bisaya["test"]))))
input_features = torch.stack(
    [torch.tensor(ex["input_features"]) for ex in sample_batch]
).to(model.device)

with torch.no_grad():
    # max_length (total sequence length, prefix included) -- not max_new_tokens,
    # which would add on top of the 4-token forced prefix and could exceed
    # Whisper's max_target_positions (448).
    generated_ids = model.generate(input_features, max_length=GENERATION_MAX_LENGTH)

pred_texts = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)
label_ids = [
    [t if t != -100 else tokenizer.pad_token_id for t in ex["labels"]]
    for ex in sample_batch
]
ref_texts = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

pd.set_option("display.max_colwidth", None)
display(pd.DataFrame({"Reference (Ground Truth)": ref_texts, "Model Prediction": pred_texts}))

torch.cuda.empty_cache()
gc.collect()


Update `kwargs` to match your run, then push the model card + final
checkpoint to the Hub. `dataset_tags`/`dataset_args` are omitted since
this corpus isn't a public Hub dataset (see "Upload the Corpus to
Kaggle" above) -- don't set them to a dataset id that doesn't exist.

In [ ]:
kwargs = {
    "dataset": "Bisaya Speech Corpus (private)",
    "language": "ceb",  # ISO 639-3 for Cebuano/Bisaya -- Hub model-card metadata only;
                         # unrelated to Whisper's own "tl" language token used above
    "model_name": "Whisper Small Bisaya",  # a 'pretty' name for our model
    "finetuned_from": MODEL_CHECKPOINT,
    "tasks": "automatic-speech-recognition",
}


In [ ]:
trainer.push_to_hub(**kwargs)


# Export for local machine

In [ ]:
# Save the model and processor explicitly from the trainer
save_directory = "./whisper_bisaya_final"

trainer.save_model(save_directory)
processor.save_pretrained(save_directory)

print("Saved model and processor to:", save_directory)


In [ ]:
import shutil
from IPython.display import FileLink

# 1. Zip the output folder
shutil.make_archive("whisper_bisaya_model", "zip", save_directory)

# 2. Display the download link
print("Click below to download your fine-tuned Whisper model:")
FileLink(r"whisper_bisaya_model.zip")


## Closing Remarks

This notebook fine-tunes Whisper small on this project's Bisaya corpus
using 🤗 Datasets, Transformers, and the Hugging Face Hub, adapted for
Kaggle's environment (GPU/internet setup, Kaggle Secrets for
authentication, dataset upload, and unattended long-run training via
Save & Run All). See the original
[fine-tuning blog post](https://huggingface.co/blog/fine-tune-whisper)
for the underlying theory, and this project's own `README.md`/`CLAUDE.md`
for how this fits alongside the Kaldi and ElevenLabs systems in the wider
benchmark.

## Evaluate on Held-Out Test Set (`compare.ipynb`-compatible)

Loads the fine-tuned model directly from the Hugging Face Hub
(`troxyz1268/whisper-small-bisaya`) -- **not** the `model`/`processor`/
`trainer` objects from the training cells above -- so this section gives
the same result whether or not training actually ran in this kernel
session. Evaluates only the same 19-utterance held-out test set (the
speaker-independent split, seed 42) that `evaluate_kaldi.ipynb` and
`evaluate_elevenlabs.ipynb` are evaluated against locally, computes raw
WER/CER, and exports `output/whisper/results.csv` in the exact schema
`evaluate_whisper.ipynb` produces locally -- download it from this
notebook's Output tab and drop it into your local `output/whisper/`
folder; `compare.ipynb` needs no changes to read it.

### Load Model (from the Hub, not the kernel's trained objects)

In [9]:
!pip install -q jiwer

import os
import time
from pathlib import Path

import pandas as pd
import torch
from transformers import pipeline

RESULTS_DIR = Path("/kaggle/working/output")
(RESULTS_DIR / "whisper").mkdir(parents=True, exist_ok=True)

# Loaded fresh from the Hub -- intentionally NOT the in-kernel model/processor
# from the training cells above, so this section is correct even if training
# didn't run in this session.
WHISPER_MODEL_ID = "troxyz1268/whisper-small-bisaya"
LANGUAGE = "tl"  # matches training (Tagalog -- Whisper's closest code to Bisaya/Cebuano)

# torch.cuda.is_available() only confirms the driver sees a GPU -- it does
# NOT confirm the installed torch build has kernels compiled for this GPU's
# compute capability. If you skipped the "Prepare Environment" cells above
# (torch/torchaudio/torchvision reinstall + kernel restart) because training
# is already done, the kernel is running Kaggle's default preinstalled torch,
# which can be mismatched against the attached GPU -- confirmed here via a
# real CUDA error ("no kernel image is available for execution on the
# device") when is_available() alone said True. So actually run a tiny op
# on cuda instead of trusting is_available(), and fall back to CPU if it fails.
device = -1
if torch.cuda.is_available():
    try:
        torch.zeros(1, device="cuda") + 1
        device = 0
    except RuntimeError as e:
        print(f"CUDA is visible but not usable ({e}) -- falling back to CPU. "
              "For GPU speed, run the 'Prepare Environment' cells above "
              "(torch/torchaudio/torchvision reinstall + kernel restart) first, "
              "then re-run this cell.")

asr = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_MODEL_ID,
    chunk_length_s=30,
    stride_length_s=5,
    device=device,
)
print(f"Loaded {WHISPER_MODEL_ID} on {'GPU' if device == 0 else 'CPU'}")

CUDA is visible but not usable (CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
) -- falling back to CPU. For GPU speed, run the 'Prepare Environment' cells above (torch/torchaudio/torchvision reinstall + kernel restart) first, then re-run this cell.


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


Loaded troxyz1268/whisper-small-bisaya on CPU


### Load Corpus and Restrict to the Held-Out Test Set

In [10]:
CORPUS_DIR = Path("/kaggle/input/datasets/troymerales/bisaya-audio")
parquet_files = sorted(CORPUS_DIR.glob("*.parquet"))
assert parquet_files, f"No Parquet files found under {CORPUS_DIR} -- check the dataset is attached (Add Input)."

dfs = []
for file in parquet_files:
    temp = pd.read_parquet(file)
    parquet_num = int(file.stem.split("-")[1])
    temp["row"] = range(1, len(temp) + 1)
    temp["id"] = temp["row"].apply(lambda x: f"{parquet_num:03d}_{x:03d}")
    temp["parquet_file"] = file.name
    dfs.append(temp)

data_df = pd.concat(dfs, ignore_index=True)

# Row position in this concatenation -- must match evaluate_kaldi.ipynb's/
# evaluate_elevenlabs.ipynb's own corpus loading (same sorted-glob +
# ignore_index=True order) for compare.ipynb's join key to line up.
data_df["corpus_index"] = data_df.index
print(f"Loaded {len(data_df)} utterances from {len(parquet_files)} shard(s)")

Loaded 90 utterances from 11 shard(s)


In [11]:
# Same speaker-independent split as train_kaldi.ipynb Section 8 and the
# training cells above (by speaker, ~80/20, fixed seed 42) -- recomputed
# here from data_df directly rather than reusing the training cells'
# `test_speakers` variable, so this section stands alone.
SPLIT_SEED = 42


def speaker_independent_split(speakers, test_fraction=0.2, seed=SPLIT_SEED):
    speakers = sorted(speakers)
    shuffled = pd.Series(speakers).sample(frac=1.0, random_state=seed)
    n_test = max(1, round(len(speakers) * test_fraction))
    test_speakers = set(shuffled.iloc[:n_test])
    train_speakers = set(shuffled.iloc[n_test:])
    assert train_speakers.isdisjoint(test_speakers)
    return train_speakers, test_speakers


all_speakers = set(data_df["speaker_id"].unique())
_, test_speakers = speaker_independent_split(all_speakers)

test_df = data_df[data_df["speaker_id"].isin(test_speakers)].reset_index(drop=True)
print(f"Held-out test set: {len(test_speakers)} speakers, {len(test_df)} utterances")

Held-out test set: 3 speakers, 19 utterances


### Transcribe via Whisper

Resumable: any `id` already present in the checkpoint file is skipped.

In [12]:
CHECKPOINT_PATH = str(RESULTS_DIR / "whisper" / "whisper_raw_results.parquet")

if Path(CHECKPOINT_PATH).exists():
    checkpoint_df = pd.read_parquet(CHECKPOINT_PATH)
else:
    checkpoint_df = pd.DataFrame(
        columns=["id", "corpus_index", "parquet_file", "row", "speaker_id", "reference", "prediction"]
    )

completed_ids = set(checkpoint_df["id"])
checkpoint_records = checkpoint_df.to_dict("records")


def save_checkpoint(records):
    tmp_path = CHECKPOINT_PATH + ".tmp"
    pd.DataFrame(records).to_parquet(tmp_path, index=False)
    os.replace(tmp_path, CHECKPOINT_PATH)


print(f"Checkpoint loaded: {len(completed_ids)} samples already transcribed.")
print(f"Remaining to process: {len(test_df) - len(completed_ids)} / {len(test_df)}")

Checkpoint loaded: 0 samples already transcribed.
Remaining to process: 19 / 19


In [13]:
for _, row in test_df.iterrows():

    sample_id = row["id"]
    if sample_id in completed_ids:
        continue

    print(f"{row['parquet_file']} — id {sample_id} ({row['duration']:.1f}s)")

    t0 = time.time()
    try:
        result = asr(row["audio"]["bytes"], generate_kwargs={"language": LANGUAGE, "task": "transcribe"})
    except Exception as e:
        print(f"  FAILED on {sample_id}: {e}")
        print("  Stopping (further calls will likely fail the same way). "
              "Fix the issue and rerun this cell -- it resumes automatically.")
        break
    print(f"  done in {time.time() - t0:.1f}s")

    checkpoint_records.append({
        "id": sample_id,
        "corpus_index": row["corpus_index"],
        "parquet_file": row["parquet_file"],
        "row": row["row"],
        "speaker_id": row["speaker_id"],
        "reference": row["transcript"],
        "prediction": result["text"]
    })
    completed_ids.add(sample_id)
    save_checkpoint(checkpoint_records)

print(f"\nDone this run. Total transcribed so far: {len(checkpoint_records)} / {len(test_df)}")

test-00001-of-00011.parquet — id 001_001 (56.4s)


A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.


  done in 71.5s
test-00001-of-00011.parquet — id 001_002 (37.7s)
  done in 52.9s
test-00001-of-00011.parquet — id 001_003 (95.6s)
  done in 138.4s
test-00001-of-00011.parquet — id 001_004 (49.4s)
  done in 60.9s
test-00001-of-00011.parquet — id 001_005 (51.3s)
  done in 93.0s
test-00001-of-00011.parquet — id 001_006 (33.6s)
  done in 46.0s
test-00001-of-00011.parquet — id 001_007 (37.4s)
  done in 57.6s
test-00004-of-00011.parquet — id 004_002 (140.4s)
  done in 158.0s
test-00004-of-00011.parquet — id 004_003 (149.8s)
  done in 169.5s
test-00004-of-00011.parquet — id 004_004 (249.2s)
  done in 296.3s
test-00004-of-00011.parquet — id 004_005 (237.7s)
  done in 277.4s
test-00004-of-00011.parquet — id 004_006 (145.9s)
  done in 164.6s
test-00005-of-00011.parquet — id 005_001 (92.6s)
  done in 109.2s
test-00005-of-00011.parquet — id 005_002 (300.4s)
  done in 321.1s
test-00005-of-00011.parquet — id 005_003 (300.4s)
  done in 300.5s
test-00005-of-00011.parquet — id 005_004 (300.3s)
  done i

### Raw WER/CER and Export for `compare.ipynb`

In [14]:
from jiwer import process_words, process_characters

results_df = pd.read_parquet(CHECKPOINT_PATH)

results = []
for _, row in results_df.iterrows():
    reference, prediction = row["reference"], row["prediction"]
    word_result = process_words(reference, prediction)
    char_result = process_characters(reference, prediction)

    results.append({
        "id": row["id"],
        "WER": word_result.wer,
        "WER_sub": word_result.substitutions,
        "WER_del": word_result.deletions,
        "WER_ins": word_result.insertions,
        "WER_hits": word_result.hits,
        "CER": char_result.cer,
        "CER_sub": char_result.substitutions,
        "CER_del": char_result.deletions,
        "CER_ins": char_result.insertions,
        "CER_hits": char_result.hits,
    })

metrics_df = pd.DataFrame(results)
metrics_df.head()

,id,WER,WER_sub,WER_del,WER_ins,WER_hits,CER,CER_sub,CER_del,CER_ins,CER_hits
0,001_001,0.561644,72,2,8,72,0.158028,48,29,48,714
1,001_002,0.584071,62,4,0,47,0.160187,40,45,18,558
2,001_003,0.775665,165,16,23,82,0.330140,186,85,177,1086
3,001_004,0.566929,61,4,7,62,0.157821,49,32,32,635
4,001_005,0.732919,98,17,3,46,0.219595,84,65,46,739


In [15]:
META_COLS = [
    "speaker_id", "language", "gender", "country", "mother_tongue", "dialect",
    "os", "device", "n_words", "age_band", "native_speaker", "proficiency",
]

analysis_df = test_df[META_COLS + ["id", "corpus_index"]].merge(
    results_df[["id", "reference", "prediction"]], on="id", how="left",
).merge(
    metrics_df[["id", "WER", "WER_sub", "WER_del", "WER_ins", "WER_hits",
                 "CER", "CER_sub", "CER_del", "CER_ins", "CER_hits"]],
    on="id", how="left",
)

EXPORT_PATH = RESULTS_DIR / "whisper" / "results.csv"
analysis_df.to_csv(EXPORT_PATH, index=False)
print(f"Wrote {len(analysis_df)} rows to {EXPORT_PATH}")
analysis_df.head()

Wrote 19 rows to /kaggle/working/output/whisper/results.csv


,speaker_id,language,gender,country,mother_tongue,dialect,os,device,n_words,age_band,...,WER,WER_sub,WER_del,WER_ins,WER_hits,CER,CER_sub,CER_del,CER_ins,CER_hits
0,CEB_012,Cebuano,male,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),Linux,Mobile,146,45-59,...,0.561644,72,2,8,72,0.158028,48,29,48,714
1,CEB_012,Cebuano,male,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),Linux,Mobile,113,45-59,...,0.584071,62,4,0,47,0.160187,40,45,18,558
2,CEB_012,Cebuano,male,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),Linux,Mobile,263,45-59,...,0.775665,165,16,23,82,0.330140,186,85,177,1086
3,CEB_012,Cebuano,male,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),Linux,Mobile,127,45-59,...,0.566929,61,4,7,62,0.157821,49,32,32,635
4,CEB_012,Cebuano,male,Philippines,Tagalog / Filipino,Philippines - Manila (Tagalog),Linux,Mobile,161,45-59,...,0.732919,98,17,3,46,0.219595,84,65,46,739
